In [ ]:
import gc
from pathlib import Path
import tarfile

from huggingface_hub import hf_hub_download
from tqdm.notebook import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "field_detecter").is_dir():
    ROOT = ROOT.parent

In [ ]:
OUT_DIR = ROOT / "data" / "agvision"
EXTRACT = True
REPO = "shi-labs/Agriculture-Vision"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)

In [ ]:
print(f"Downloading {REPO} (tar.gz, ~21 GB)...")

archive = hf_hub_download(
    repo_id=REPO,
    filename="Agriculture-Vision-2021.tar.gz",
    repo_type="dataset",
    local_dir=str(OUT_DIR),
)

archive_path = Path(archive)

print("Saved:", archive_path)


In [ ]:
if EXTRACT:
    print("Extracting...")

    with tarfile.open(archive_path, "r:gz") as tf:
        for member in tqdm(tf, desc="Extracting", unit="files"):
            tf.extract(member, path=OUT_DIR)

    gc.collect()
    print(f"Extracted to {OUT_DIR}")

    expected = (
        OUT_DIR
        / "Agriculture-Vision-2021"
        / "train"
        / "images"
        / "rgb"
    )

    if expected.is_dir():
        print("OK:", expected)
    else:
        print(
            "Warning: expected "
            "Agriculture-Vision-2021/train/images/rgb "
            "— check archive layout."
        )

In [ ]:
SOURCE_DIR = OUT_DIR / "Agriculture-Vision-2021"
ARCHIVE_NAME = OUT_DIR / "Agriculture-Vision-2021-processed.tar.gz"

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Directory not found: {SOURCE_DIR}")

print(f"Packing {SOURCE_DIR}...")
print(f"Output archive: {ARCHIVE_NAME}")


def iter_dataset_files(root: Path):
    for path in root.rglob("*"):
        if path.is_file():
            yield path


with tarfile.open(ARCHIVE_NAME, "w:gz") as tf:
    for path in tqdm(iter_dataset_files(SOURCE_DIR), desc="Packing", unit="files"):
        tf.add(path, arcname=path.relative_to(SOURCE_DIR.parent))

gc.collect()
print("Done.")
print(f"Archive saved to: {ARCHIVE_NAME}")